# Chapter 3 — Multimodal Fusion

This notebook builds the **joint multimodal input sequence** for nanochat_vlm.

Goal:
    [image patches] + [text tokens] → single LM input

This notebook:
- prepends image tokens
- constructs embeddings
- defines attention masks

No training happens here.

## Fusion Strategy: Prepend Image Tokens

We use a **prepend strategy**:

    [<im_start>, P1, P2, ..., Pn, <im_end>] + text

Reasons:
- simple
- widely used (LLaVA-style)
- causal masking remains valid

## Symbol-Level Sequence

Token sequence (conceptual):

    <bos>
    <im_start>
    <im_patch> × N
    <im_end>
    text tokens...

In [ ]:
import torch

## Dimensions

These must match previous notebooks.

In [ ]:
BATCH = 2
NUM_PATCHES = 256
LM_DIM = 2048
TEXT_LEN = 16

## Inputs

We assume:
- projected vision embeddings are ready
- text token embeddings are ready

In [ ]:
# Vision (after projector)
vision_embeds = torch.randn(BATCH, NUM_PATCHES, LM_DIM)

# Text token embeddings (from LM embedding table)
text_embeds = torch.randn(BATCH, TEXT_LEN, LM_DIM)

## Special Token Embeddings

<im_start> and <im_end> are learned embeddings
from the LM embedding table.

In [ ]:
im_start_embed = torch.randn(1, 1, LM_DIM)
im_end_embed   = torch.randn(1, 1, LM_DIM)

im_start_embed = im_start_embed.expand(BATCH, -1, -1)
im_end_embed   = im_end_embed.expand(BATCH, -1, -1)